# 05 · General Linear Model (GLM)  (⏱ ~90 min)

**Concept (the classic fMRI analysis, formerly the separate `nistats` package, now `nilearn.glm`):**

1. **Design matrix:** your task events (when stimuli happened) convolved with a model of the *hemodynamic response function (HRF)*, plus nuisance regressors (drift, motion).
2. **Fit** at every voxel: `signal = design_matrix @ betas + noise`.
3. **Contrast:** a weighted combination of betas that asks a question (e.g. `auditory - rest`) -> a z/t map.
4. **Threshold correctly:** you're testing ~100k voxels, so you need multiple-comparison correction (FDR, Bonferroni, cluster-level).
5. **Second level:** combine many subjects' contrast maps with another GLM (usually a one-sample test) to make group-level claims.

Key objects: `nilearn.glm.first_level.FirstLevelModel`, `make_first_level_design_matrix`, `nilearn.glm.second_level.SecondLevelModel`, `nilearn.glm.threshold_stats_img`.

**Gallery to read after your attempt:**
- Start with `00_tutorials/plot_single_subject_single_run.py`: it is this exact dataset (SPM auditory), start to finish.
- `04_glm_first_level`: `plot_design_matrix.py`, `plot_hrf.py`, `plot_first_level_details.py`, `plot_two_runs_model.py`, `plot_bids_features.py`
- `05_glm_second_level`: `plot_second_level_one_sample_test.py` (Part B), `plot_second_level_two_sample_test.py`, `plot_thresholding.py` (compares the correction methods from Ex 4)

In [ ]:
!pip install -q nilearn
import nilearn, numpy as np, pandas as pd, matplotlib.pyplot as plt
print("nilearn", nilearn.__version__)

**Data tip:** nilearn downloads datasets to `~/nilearn_data`, which Colab wipes when the runtime resets.
If you don't want to re-download, mount Drive and pass `data_dir=` to the `fetch_*` functions
(`from google.colab import drive; drive.mount('/content/drive')`).

In [ ]:
from nilearn import datasets, plotting
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.glm import threshold_stats_img
spm = datasets.fetch_spm_auditory()    # one subject, block design: listening vs rest
print(spm.keys())

## Part A: first level (one subject)
- **Ex 1.** The dataset bunch gives you `spm.func`, `spm.events` (a TSV path) and `spm.t_r`. The events table needs `onset`, `duration`, `trial_type`: load it with `pandas.read_table` and look at it. What is the TR (surprisingly long! think about what that means for how finely you can resolve the response), how many scans are there, and how many events?
- **Ex 2.** Build a design matrix yourself with `make_first_level_design_matrix(frame_times, events, hrf_model="spm", drift_model="cosine")` (you'll need to construct `frame_times` from the scan count and TR), then `plotting.plot_design_matrix`. *Predict:* why does the task column look smoothed/delayed compared to the raw events? What are the other columns? (Later you'll notice `FirstLevelModel.fit(img, events)` builds this for you, stored in `design_matrices_`; compare yours with it.)
- **Ex 3.** Fit a `FirstLevelModel(t_r=..., hrf_model="spm", ...)`. Then `compute_contrast("listening", output_type="z_score")`. Plot with `plot_stat_map`. If the contrast name errors, print the design matrix columns: contrast names must match them.
- **Ex 4.** Threshold properly: `threshold_stats_img(z_map, alpha=0.05, height_control="fdr")`. Careful: it returns *two* things (the thresholded image and the threshold value used). Compare with an uncorrected `threshold=3.1`. How many more voxels survive uncorrected? Why is that bad news, not good?
- **Ex 5.** Change `hrf_model` (e.g. `"spm + derivative"`) or `drift_model`. How does the map change? What does the derivative term allow for?

## Part B: second level (group)
Use `datasets.fetch_localizer_contrasts(["left vs right button press"], n_subjects=16)` (or the one used by the gallery's second-level examples).
- **Ex 6.** Build a design matrix for a **one-sample test**: a single column of ones (an intercept) with one row per subject (`pandas`). Fit a `SecondLevelModel` on the list of per-subject contrast maps and `compute_contrast("intercept")`.
- **Ex 7.** Threshold with FDR or cluster correction and plot. Does the activation land in motor cortex (contralateral, since it's left vs right)?
- **Ex 8 (stretch).** Two-sample comparison using the phenotypic table (e.g. between groups), or add a covariate such as age. What changes in the design matrix?

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Ex 5

In [ ]:
# Ex 6

In [ ]:
# Ex 7

## Check your understanding
- Why convolve events with an HRF instead of using 0/1 boxcars directly?
- A contrast vector `[1, -1, 0, ...]` asks what question?
- Why is a group analysis done on contrast maps from each subject rather than by pooling all scans into one giant model?

## Industry/research angle
This is the workhorse of cognitive-neuroscience papers, pre-surgical language/motor mapping, and pharmacological imaging trials. In research, knowing *why* each step exists is how you review (and trust) results.